# 03 — Training Data Preparation

Phase 6-7. Converts the image index into multimodal instruction examples,
using a **leakage-safe group split**.


In [ ]:
# --- Colab setup (skip if running locally) ---
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('casting-defect-vlm'):
        # Replace with your repository URL, or upload the folder to Colab.
        raise SystemExit('Upload the casting-defect-vlm project folder to Colab first.')
    %cd casting-defect-vlm
    !pip install -q -r requirements.txt

sys.path.insert(0, os.path.abspath('..' if os.path.basename(os.getcwd())=='notebooks' else '.'))
print('python', sys.version.split()[0], '| colab:', IN_COLAB)


## 1. Why a group split

The defect images were generated *from* the OK images, so one source
casting appears as an `ok/` image and as several defective derivatives.
A plain stratified split would put a casting in train and its own
derivative in test — leaking the answer.

We therefore split by inferred source casting, and `verify_no_leakage()`
**raises** rather than warns.


In [ ]:
!python scripts/prepare_training_data.py --inspect-groups


## 2. Build the splits and JSONL files


In [ ]:
!python scripts/prepare_training_data.py


## 3. Split statistics


In [ ]:
import json, pandas as pd
from pathlib import Path

stats = json.loads(Path('results/training/dataset_statistics.json').read_text())
print('seed:', stats['seed'], '| ratios:', stats['split_ratios'])
for s in ['train','validation','test']:
    d = stats[s]
    print(f"{s:<12}{d['n_examples']:>6} examples | {d['n_unique_groups']:>4} groups | {d['binary_distribution']}")


## 4. Leakage verification


In [ ]:
print(json.dumps(stats['leakage_check'], indent=2))
print(json.dumps(stats['duplicate_removal'], indent=2))


## 5. Inspect an example

Answers are derived strictly from ground-truth labels — nothing invented.


In [ ]:
from src.prepare_data import load_jsonl
train = load_jsonl('data/processed/train.jsonl')
ex = train[0]
print('image      :', ex['relpath'])
print('truth      :', ex['ground_truth'], '| defect type:', ex['defect_type'])
print('question   :', ex['messages'][0]['content'][1]['text'])
print('answer     :'); print(ex['messages'][1]['content'][0]['text'])


## 6. Verify no contradictory examples


In [ ]:
bad = [e for e in train
       if (e['ground_truth']=='OK') != ('Classification: OK' in e['messages'][1]['content'][0]['text'])]
print('contradictory examples:', len(bad))
assert not bad, 'training answers disagree with ground-truth labels'


## 7. Instruction template coverage


In [ ]:
import collections
c = collections.Counter(e['messages'][0]['content'][1]['text'] for e in train)
for k, v in c.most_common(): print(f'{v:>5}  {k}')
